In [0]:
file_path = "/Volumes/data3/default/pd5201_klastry/SRR16356247_1.fastq"

lines_df = spark.read.text(file_path)

lines_df.show(8, truncate=False)

print(f"Liczba linii: {lines_df.count()}")
print(f"Liczba odczytów: {lines_df.count() // 4}")

+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                  |
+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|@SRR16356247.1 1 length=151                                                                                                                            |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|
|+SRR16356247.1 1 length=151                                                                                                                            |
|A=AFFFFFFFFFFFFFF#F/FAFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF#FFFFFFFFFF#FFFFF

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id

window = Window.orderBy(monotonically_increasing_id())

lines_with_id = lines_df.withColumn(
    "line_number",
    row_number().over(window) - 1
)

lines_with_id.show(8, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|value                                                                                                                                                  |line_number|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |
|+SRR16356247.1 1 length=151                                                                                                                            |2          |
|A=A

In [0]:
from pyspark.sql.functions import when, col

lines_typed = lines_with_id.withColumn(
    "line_type",
    when(col("line_number") % 4 == 0, "header")
    .when(col("line_number") % 4 == 1, "sequence")
    .when(col("line_number") % 4 == 2, "separator")
    .when(col("line_number") % 4 == 3, "quality")
)

lines_typed.show(12, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|value                                                                                                                                                  |line_number|line_type|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |
|+SRR16356247.1 1 length=151                                                                                            

In [0]:
lines_with_record = lines_typed.withColumn(
    "record_id",
    (col("line_number") / 4).cast("integer")
)

lines_with_record.show(16, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|value                                                                                                                                                  |line_number|line_type|record_id|
+-------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+---------+---------+
|@SRR16356247.1 1 length=151                                                                                                                            |0          |header   |0        |
|GCTCCCAACCAAGCTCTNTTGAGGATCTTGAAGGAAACTGAATTCAAAAAGATCAAAGNGCTGGGCTCCNGTGCGTTCGGCACGGTGTATAAGGTAAGGTCCCTGGCACAGGCCTCTGGGCTGGGCCGCAGGGCCTCTCATGGTCTGGTGG|1          |sequence |0        |
|+SRR16356247.1 1 length=151                                          

In [0]:
from pyspark.sql.functions import first

fastq_wide = lines_with_record.groupBy("record_id").pivot("line_type").agg(
    first("value")
)

fastq_wide.show(5, truncate=False)
fastq_wide.printSchema()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|header                     |quality                                                                                                                                                |separator                  |sequence                                                                                                                                               |
+---------+---------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+-------------------------------------

In [0]:
from pyspark.sql.functions import col, regexp_replace, split

fastq_clean = fastq_wide.withColumn(
    "read_id",
    split(
        regexp_replace(col("header"), "^@", ""),
        " "
    )[0]
)

fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
).show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        

In [0]:
fastq_clean = fastq_clean.select(
    "record_id",
    "read_id",
    "sequence",
    "quality"
)

fastq_clean.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|record_id|read_id      |sequence                                                                                                                                               |quality                                                                                                                                                |
+---------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        